# Integração numérica

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/11-integracao.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. deduzir as regras do trapézio e de Simpson e prever a ordem do erro;
> 2. acelerar a convergência com Romberg e usar quadraturas de Gauss;
> 3. entender o que uma rotina adaptativa como `scipy.integrate.quad` faz, e quando ela erra calada;
> 4. tratar integrandos singulares com mudança de variável;
> 5. usar Monte Carlo em muitas dimensões, com a barra de erro certa.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
from scipy import integrate
from cnlivro.integracao import (trapezio, simpson, trapezio_dados, gauss_legendre, romberg,
                                simpson_adaptativo, monte_carlo)

## 1. Problema de engenharia: quanto tempo para esvaziar um tanque?

Um tanque de decantação tem a forma de um tronco de cone com fundo plano: raio $r_0 = 1$ m no fundo, $r_1 = 2$ m no topo e altura $H = 3$ m. Entre duas bateladas, ele é esvaziado por um orifício no fundo, de diâmetro $d = 5$ cm e coeficiente de descarga $C_d = 0{,}61$. A operação precisa saber quanto tempo isso leva, partindo do tanque cheio.

Pela lei de Torricelli, a vazão pelo orifício é $Q = C_d\, a \sqrt{2gh}$, com $a = \pi d^2/4$ e $h$ o nível da água. O volume que sai é o que o nível desce: $A(h)\,dh = -Q\,dt$, com $A(h)$ a área da seção na altura $h$. Separando as variáveis,

$$
T = \int_0^H \frac{A(h)}{C_d\, a\sqrt{2g}}\,\frac{dh}{\sqrt{h}},
\qquad A(h) = \pi\left(r_0 + (r_1 - r_0)\frac{h}{H}\right)^2 .
$$ (eq-tanque)

Para essa forma há primitiva, e o resultado exato servirá para conferir. Para um tanque de forma qualquer, dada por uma **tabela de arqueação** (área em função do nível, medida na obra), só resta integrar numericamente. Repare no fator $1/\sqrt{h}$: o integrando é **infinito** em $h = 0$, embora a integral seja finita.

In [ ]:
g, r0, r1, H = 9.81, 1.0, 2.0, 3.0          # m/s², m, m, m
d, Cd = 0.05, 0.61                          # m, adimensional
k = Cd * np.pi * d**2 / 4 * np.sqrt(2 * g)  # m^(5/2)/s

def area(h):
    return np.pi * (r0 + (r1 - r0) * h / H) ** 2

def integrando(h):
    return area(h) / (k * np.sqrt(h))

# primitiva de π (r0 + c h)² / √h, termo a termo
c = (r1 - r0) / H
T_exato = np.pi / k * (2 * r0**2 * H**0.5 + 4 / 3 * r0 * c * H**1.5 + 2 / 5 * c**2 * H**2.5)
print(f"tempo exato de esvaziamento: {T_exato:.1f} s = {T_exato / 60:.2f} min")

In [ ]:
hs = np.linspace(1e-3, H, 600)
fig = go.Figure()
fig.add_trace(go.Scatter(x=hs, y=integrando(hs), mode="lines", fill="tozeroy", name="A(h) / (C_d a √(2g h))",
                         line=dict(color=CORES["aproximacao"]), fillcolor="rgba(0,114,178,0.15)"))
fig.update_xaxes(title="nível h (m)")
fig.update_yaxes(title="integrando (s/m)", range=[0, 6000])
fig.update_layout(width=760, height=400, title="A área sob a curva é o tempo de esvaziamento", showlegend=False)
mostrar(fig)

**O que notar:** perto do topo, o integrando é quase constante. Perto do fundo, ele dispara para o infinito: com pouca água, a pressão é pequena, a vazão também, e cada centímetro de nível demora muito. A área sob esse pico é finita, mas uma regra que avalie o integrando em $h = 0$ tem um problema.

## 2. A ideia em uma imagem

Integrar é calcular a área sob uma curva. As regras de Newton-Cotes trocam a curva, em cada pedaço, por algo que sabemos integrar: retas (trapézio) ou parábolas (Simpson). Arraste o slider para aumentar o número de subintervalos $n$.

In [ ]:
f2 = lambda x: 1.2 + 0.8 * np.sin(2.2 * x) * np.exp(-0.25 * x) + 0.15 * x
a2, b2 = 0.0, 4.0
exato2 = integrate.quad(f2, a2, b2)[0]
xx = np.linspace(a2, b2, 400)

def quadro_nc(n):
    # títulos fixos nos painéis; os erros vão no título geral, que muda com o slider
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=["trapézio", "Simpson"])
    xn = np.linspace(a2, b2, n + 1)
    # trapézio: polígono pelos nós
    fig.add_trace(go.Scatter(x=np.r_[xn, xn[::-1]], y=np.r_[f2(xn), np.zeros(n + 1)], fill="toself", mode="lines",
                             fillcolor="rgba(213,94,0,0.18)", line=dict(color=CORES["erro"], width=1.5),
                             name="aproximação"), 1, 1)
    # Simpson: parábola por três nós em cada par de subintervalos
    xs_p, ys_p = [], []
    for i in range(0, n, 2):
        xa, xm, xb = xn[i], xn[i + 1], xn[i + 2]
        p = np.polyfit([xa, xm, xb], f2(np.array([xa, xm, xb])), 2)
        xl = np.linspace(xa, xb, 40)
        xs_p += list(xl)
        ys_p += list(np.polyval(p, xl))
    xs_p, ys_p = np.array(xs_p), np.array(ys_p)
    fig.add_trace(go.Scatter(x=np.r_[xs_p, xs_p[::-1]], y=np.r_[ys_p, np.zeros_like(ys_p)], fill="toself", mode="lines",
                             fillcolor="rgba(213,94,0,0.18)", line=dict(color=CORES["erro"], width=1.5),
                             showlegend=False), 1, 2)
    for col in (1, 2):
        fig.add_trace(go.Scatter(x=xx, y=f2(xx), mode="lines", line=dict(color=CORES["exata"], width=2),
                                 name="f(x)", showlegend=(col == 1)), 1, col)
        fig.add_trace(go.Scatter(x=xn, y=f2(xn), mode="markers", marker=dict(color=CORES["destaque"], size=8),
                                 name="nós", showlegend=(col == 1)), 1, col)
    fig.update_xaxes(title="x")
    fig.update_yaxes(range=[0, 2.6])
    fig.update_layout(width=980, height=450, legend=dict(orientation="h", y=-0.2), margin=dict(t=110),
                      title=f"n = {n}: erro do trapézio {numero(trapezio(f2, a2, b2, n) - exato2, 4)}, "
                            f"erro de Simpson {numero(simpson(f2, a2, b2, n) - exato2, 4)}")
    return fig

quadros = [(str(n), quadro_nc(n)) for n in [2, 4, 8, 16]]
mostrar(com_controle(quadros, rotulo="n = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com o mesmo número de nós, as parábolas acompanham a curva muito melhor que as retas. Dobrar $n$ divide o erro do trapézio por cerca de 4, e o de Simpson por cerca de 16. Com $n = 16$, a aproximação de Simpson já é indistinguível da curva.

## 3. Formulação matemática

### 3.1 Regras de Newton-Cotes

Dividimos $[a, b]$ em $n$ subintervalos de largura $h = (b - a)/n$, com nós $x_i = a + ih$ e valores $f_i = f(x_i)$.

**Trapézio.** Em cada subintervalo, a área de um trapézio. Somando,

$$
T_n = h\left[\frac{f_0}{2} + f_1 + \cdots + f_{n-1} + \frac{f_n}{2}\right],
\qquad
\int_a^b f\,dx - T_n = -\frac{(b - a)\,h^2}{12}f''(\xi).
$$ (eq-trapezio)

O erro é de **ordem 2** e exato para retas.

**Simpson 1/3.** Uma parábola por cada trio de nós (por isso $n$ precisa ser **par**):

$$
S_n = \frac{h}{3}\left[f_0 + 4f_1 + 2f_2 + 4f_3 + \cdots + 4f_{n-1} + f_n\right],
\qquad
\int_a^b f\,dx - S_n = -\frac{(b - a)\,h^4}{180}f^{(4)}(\xi).
$$ (eq-simpson)

O erro é de **ordem 4**. Por sorte de simetria, a regra é exata para cúbicas, e não só para parábolas. A regra 3/8 de Simpson usa cúbicas por quatro nós, com a mesma ordem; serve para completar um número ímpar de subintervalos.

**Dados tabelados.** Com espaçamento irregular, o trapézio continua simples: $\sum_i (x_{i+1} - x_i)(y_i + y_{i+1})/2$, que é `np.trapezoid(y, x)`. Para Simpson com espaçamento irregular, `scipy.integrate.simpson(y, x=x)`.

As fórmulas de erro só valem se as derivadas que aparecem nelas **existirem e forem limitadas**. Em {eq}`eq-tanque`, $f''$ e $f^{(4)}$ são infinitas em $h = 0$, e a ordem prometida não se realiza (seção 6).

### 3.2 Romberg: Richardson aplicado ao trapézio

Para $f$ suave, o erro do trapézio tem a forma $c_2 h^2 + c_4 h^4 + c_6 h^6 + \cdots$, só com potências pares (fórmula de Euler-Maclaurin). Como no capítulo 10, combinando $T_n$ e $T_{2n}$ elimina-se o primeiro termo:

$$
R_{k,1} = \frac{4T_{2n} - T_n}{3}, \qquad
R_{k,j} = R_{k,j-1} + \frac{R_{k,j-1} - R_{k-1,j-1}}{4^j - 1}.
$$

A primeira coluna extrapolada é exatamente a regra de Simpson. Cada nível dobra os nós, reaproveitando os anteriores, e ganha duas ordens.

### 3.3 Quadratura de Gauss

Newton-Cotes fixa os nós igualmente espaçados e escolhe os pesos. Gauss escolhe **nós e pesos**: com $n$ pontos há $2n$ parâmetros, e é possível acertar exatamente polinômios de grau até $2n - 1$:

$$
\int_{-1}^{1} f(t)\,dt \approx \sum_{i=1}^{n} w_i f(t_i).
$$

Os nós $t_i$ são as raízes do polinômio de Legendre de grau $n$ e ficam **dentro** do intervalo, sem tocar nas pontas. Para dois pontos, $t = \pm 1/\sqrt3$ e $w = 1$. Para outro intervalo, troca-se a variável: $x = \frac{a + b}{2} + \frac{b - a}{2}t$. Para funções suaves, o erro cai **exponencialmente** com $n$.

### 3.4 Integração adaptativa

Funções reais costumam ser suaves em quase todo o intervalo, com poucas regiões difíceis. Uma rotina **adaptativa** estima o erro em cada pedaço e só subdivide onde é preciso. No Simpson adaptativo, compara-se $S$ no pedaço inteiro com $S$ nas duas metades: pela fórmula {eq}`eq-simpson`, o erro da soma das metades é cerca de $|S_{\text{metades}} - S_{\text{inteiro}}|/15$. Se for maior que a tolerância, cada metade é tratada da mesma forma.

`scipy.integrate.quad` usa a biblioteca QUADPACK: regras de Gauss-Kronrod de 21 pontos, subdivisão adaptativa e extrapolação, que também lida bem com singularidades nas pontas do tipo $1/\sqrt{x}$. É a escolha padrão para funções que você pode avaliar.

### 3.5 Monte Carlo e a maldição da dimensão

Uma regra de produto com $n$ pontos por eixo, em $d$ dimensões, usa $N = n^d$ pontos. Com erro $O(n^{-p})$ por eixo, o erro em função do custo é $O(N^{-p/d})$: em dimensão alta, nenhuma ordem $p$ salva.

**Monte Carlo** sorteia $N$ pontos uniformes numa região de volume $V$ e usa a média:

$$
\int_\Omega f\,d\mathbf{x} \approx V\,\bar f, \qquad \text{erro padrão} \approx \frac{V\,s_f}{\sqrt{N}},
$$ (eq-mc)

com $s_f$ o desvio padrão amostral dos valores de $f$. O erro cai como $N^{-1/2}$ **em qualquer dimensão**. É lento em uma dimensão e imbatível em dez.

## 4. À mão

Vamos calcular $I = \int_0^1 e^{-x^2}\,dx = 0{,}746824$, uma integral sem primitiva elementar.

| $x$ | $0$ | $0{,}25$ | $0{,}5$ | $0{,}75$ | $1$ |
| --- | --- | --- | --- | --- | --- |
| $e^{-x^2}$ | $1$ | $0{,}939413$ | $0{,}778801$ | $0{,}569783$ | $0{,}367879$ |

- **Trapézio, $n = 1$:** $\frac{1}{2}(1 + 0{,}367879) = 0{,}683940$. Erro de $0{,}063$.
- **Trapézio, $n = 2$:** $0{,}5\left[\frac{1}{2} + 0{,}778801 + \frac{0{,}367879}{2}\right] = 0{,}731370$. Erro de $0{,}015$: dividido por 4.
- **Simpson, $n = 2$:** $\frac{0{,}5}{3}(1 + 4 \times 0{,}778801 + 0{,}367879) = 0{,}747180$. Erro de $0{,}00036$.
- **Romberg:** $(4 \times 0{,}731370 - 0{,}683940)/3 = 0{,}747180$. É o mesmo número de Simpson, como prometido na seção 3.2.
- **Gauss com 2 pontos:** nós $0{,}5 \pm 0{,}5/\sqrt3 = 0{,}211325$ e $0{,}788675$, com pesos $0{,}5$: $0{,}5(0{,}956355 + 0{,}536837) = 0{,}746595$. Erro de $0{,}00023$, **menor que o de Simpson com só 2 avaliações** em vez de 3.

In [ ]:
f_hand = lambda x: np.exp(-x**2)
I_ref = integrate.quad(f_hand, 0, 1)[0]
for nome, valor in [("trapézio n=1", trapezio(f_hand, 0, 1, 1)), ("trapézio n=2", trapezio(f_hand, 0, 1, 2)),
                    ("Simpson n=2", simpson(f_hand, 0, 1, 2)), ("Gauss 2 pontos", gauss_legendre(f_hand, 0, 1, 2)),
                    ("Gauss 4 pontos", gauss_legendre(f_hand, 0, 1, 4))]:
    print(f"{nome:15s} = {valor:.6f}   erro = {valor - I_ref:+.1e}")

## 5. Explore

### 5.1 Erro em função do custo

O custo de uma integração é o número de avaliações de $f$. Para $\int_0^1 \frac{4}{1 + x^2}\,dx = \pi$:

In [ ]:
f_pi = lambda x: 4 / (1 + x**2)
ns = 2 ** np.arange(1, 11)
dados = {
    "trapézio": ([n + 1 for n in ns], [abs(trapezio(f_pi, 0, 1, n) - np.pi) for n in ns]),
    "Simpson": ([n + 1 for n in ns], [abs(simpson(f_pi, 0, 1, n) - np.pi) for n in ns]),
    "Gauss-Legendre": (list(range(1, 13)), [abs(gauss_legendre(f_pi, 0, 1, n) - np.pi) for n in range(1, 13)]),
}
rr = romberg(f_pi, 0, 1, tol=1e-15, max_niveis=8)
dados["Romberg"] = ([2**i + 1 for i in range(len(rr.historico))], [abs(v - np.pi) for v in rr.historico])
cores = {"trapézio": CORES["erro"], "Simpson": CORES["aproximacao"], "Gauss-Legendre": CORES["destaque"],
         "Romberg": CORES["secundaria"]}
fig = go.Figure()
for nome, (nav, err) in dados.items():
    err = np.maximum(err, 1e-17)
    fig.add_trace(go.Scatter(x=nav, y=err, mode="lines+markers", name=nome, line=dict(color=cores[nome])))
fig.update_xaxes(type="log", title="avaliações de f")
fig.update_yaxes(type="log", title="erro absoluto", exponentformat="power")
fig.update_layout(width=780, height=460, title="Integral de 4/(1 + x²) em [0, 1]", legend=dict(x=0.75, y=0.98))
mostrar(fig)

**O que notar:** em escala log-log, o trapézio e Simpson são retas de inclinação $-2$ e $-4$. Gauss e Romberg **curvam para baixo**: o erro cai mais depressa que qualquer potência. Gauss chega à precisão de máquina com 12 avaliações, e Romberg com pouco mais de cem, enquanto o trapézio, com mil avaliações, ainda erra na sétima casa. Para funções suaves que você pode avaliar onde quiser, regras de ordem alta valem muito a pena.

### 5.2 A surpresa do trapézio: funções periódicas

Para integrar uma função **periódica** num período inteiro, o trapézio, a regra mais simples, é a melhor de todas:

In [ ]:
f_per = lambda x: 1 / (2 + np.cos(x))                  # ∫₀^{2π} = 2π/√3
exato_per = 2 * np.pi / np.sqrt(3)
ns = np.array([4, 8, 12, 16, 24, 32, 48, 64])
e_trap = [abs(trapezio(f_per, 0, 2 * np.pi, n) - exato_per) for n in ns]
e_simp = [abs(simpson(f_per, 0, 2 * np.pi, n) - exato_per) for n in ns]
fig = go.Figure()
fig.add_trace(go.Scatter(x=ns, y=np.maximum(e_trap, 1e-17), mode="lines+markers", name="trapézio",
                         line=dict(color=CORES["erro"])))
fig.add_trace(go.Scatter(x=ns, y=np.maximum(e_simp, 1e-17), mode="lines+markers", name="Simpson",
                         line=dict(color=CORES["aproximacao"])))
fig.update_xaxes(title="número de subintervalos n")
fig.update_yaxes(type="log", title="erro absoluto", exponentformat="power")
fig.update_layout(width=760, height=420, title="Integral de 1/(2 + cos x) num período (eixo x linear)")
mostrar(fig)

**O que notar:** com o eixo $n$ **linear**, o erro do trapézio é uma reta: ele cai **exponencialmente**, e chega a $10^{-16}$ com 32 pontos, enquanto Simpson ainda está em $10^{-9}$. Numa função periódica, todos os termos de Euler-Maclaurin se cancelam entre as duas pontas. Valores eficazes de correntes, potências médias em ciclos de motores e coeficientes de Fourier são integrais desse tipo: aí, o trapézio é a regra certa, e "melhorar" para Simpson piora.

### 5.3 Onde uma rotina adaptativa trabalha

Arraste o slider para mudar a tolerância do Simpson adaptativo aplicado a uma função com dois picos.

In [ ]:
f_picos = lambda x: 1 / ((x - 0.3) ** 2 + 0.01) + 1 / ((x - 0.9) ** 2 + 0.04) - 6
exato_picos = integrate.quad(f_picos, 0, 1, epsabs=1e-13)[0]
xx = np.linspace(0, 1, 600)

def quadro_adapt(tol):
    r = simpson_adaptativo(f_picos, 0.0, 1.0, tol=tol)
    bordas = sorted({c_ for iv in r.intervalos for c_ in iv})
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xx, y=f_picos(xx), mode="lines", name="f(x)", line=dict(color=CORES["exata"])))
    fig.add_trace(go.Scatter(x=bordas, y=np.full(len(bordas), -15), mode="markers", name="bordas dos subintervalos",
                             marker=dict(symbol="line-ns-open", size=14, color=CORES["aproximacao"])))
    fig.update_xaxes(title="x")
    fig.update_yaxes(title="f(x)", range=[-22, 100])
    fig.update_layout(width=780, height=430, legend=dict(x=0.6, y=0.98),
                      title=f"tol = {tol:.0e}: {r.avaliacoes} avaliações, erro = {abs(r.valor - exato_picos):.1e}"
                      .replace(".", ","))
    return fig

tols = [1e-2, 1e-4, 1e-6, 1e-8]
quadros = [(f"{t_:.0e}", quadro_adapt(t_)) for t_ in tols]
mostrar(com_controle(quadros, rotulo="tolerância = ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** as bordas se acumulam nos dois picos, onde a curvatura é grande, e ficam esparsas no resto. Cada fator 100 na tolerância custa um fator de cerca de 3 nas avaliações, como esperado para ordem 4 ($100^{1/4} \approx 3{,}2$). Uma regra com passo uniforme precisaria do passo mais fino em todo o intervalo.

### 5.4 Monte Carlo contra a maldição da dimensão

Para $\int_{[0,1]^d} \prod_i \cos x_i \, d\mathbf{x} = (\sin 1)^d$, comparamos o trapézio em grade ($n$ pontos por eixo, $N = n^d$ avaliações) com Monte Carlo ($N$ pontos sorteados), em 1 e em 6 dimensões.

In [ ]:
rng = np.random.default_rng(0)
fig = make_subplots(rows=1, cols=2, subplot_titles=["d = 1", "d = 6"], horizontal_spacing=0.1)
for col, dim in [(1, 1), (2, 6)]:
    exato_d = np.sin(1.0) ** dim
    n_eixo = np.unique(np.round(np.logspace(0.3, 6 / dim, 12)).astype(int))
    N_grade = n_eixo**dim
    # a integral de um produto, numa grade produto, é o produto das integrais 1D
    e_grade = [abs(trapezio(np.cos, 0, 1, n - 1) ** dim - exato_d) for n in n_eixo if n > 1]
    fig.add_trace(go.Scatter(x=N_grade[n_eixo > 1], y=e_grade, mode="lines+markers", name="trapézio em grade",
                             line=dict(color=CORES["erro"]), showlegend=(col == 1)), 1, col)
    N_mc = np.logspace(2, 6, 9).astype(int)
    e_mc, barra = [], []
    for N in N_mc:
        r = monte_carlo(lambda p: np.prod(np.cos(p), axis=1), [(0, 1)] * dim, N, rng)
        e_mc.append(abs(r.valor - exato_d))
        barra.append(r.erros[0])
    fig.add_trace(go.Scatter(x=N_mc, y=e_mc, mode="markers", name="Monte Carlo (erro real)",
                             marker=dict(color=CORES["aproximacao"], size=8), showlegend=(col == 1)), 1, col)
    fig.add_trace(go.Scatter(x=N_mc, y=barra, mode="lines", name="Monte Carlo (erro padrão estimado)",
                             line=dict(color=CORES["aproximacao"], dash="dash"), showlegend=(col == 1)), 1, col)
fig.update_xaxes(type="log", title="avaliações N", exponentformat="power")
fig.update_yaxes(type="log", title="erro", exponentformat="power")
fig.update_layout(width=980, height=440, legend=dict(orientation="h", y=-0.22))
mostrar(fig)

**O que notar:** em uma dimensão, o trapézio ganha de longe (inclinação $-2$ contra $-1/2$). Em seis dimensões, o trapézio cai só como $N^{-1/3}$, e Monte Carlo, com a mesma inclinação $-1/2$ de antes, empata ou ganha. Repare também que o erro real de Monte Carlo oscila em torno do **erro padrão estimado** {eq}`eq-mc`: a barra de erro de Monte Carlo é uma estimativa estatística, e é ela que se deve relatar junto com o resultado. Em dimensões ainda maiores, como no cálculo de probabilidades de falha com dezenas de variáveis incertas (apêndice A), Monte Carlo é praticamente a única opção.

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Calcule o tempo de esvaziamento do tanque pela integral de Torricelli, usando a regra de Simpson composta com 1000 subintervalos. Estime o erro comparando com 500 subintervalos e diga quantos algarismos do resultado são confiáveis."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def tempo_esvaziamento(n=1000):
    h = np.linspace(0.0, H, n + 1)
    with np.errstate(divide="ignore"):
        y = area(h) / (k * np.sqrt(h))
    y[0] = 0.0          # em h = 0 o integrando é infinito; a contribuição de um ponto é desprezível
    passo = H / n
    return passo / 3 * (y[0] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum() + y[-1])

T_1000 = tempo_esvaziamento(1000)
T_500 = tempo_esvaziamento(500)
erro_est = abs(T_1000 - T_500) / 15          # Richardson para Simpson, ordem 4
print(f"tempo de esvaziamento: {T_1000 / 60:.3f} min")
print(f"erro estimado: {erro_est:.2f} s  ->  resultado confiável até {erro_est / 60:.3f} min")

O código é limpo, a fórmula de Simpson está correta, e a estimativa de erro usa Richardson, como na seção 3.2. O resultado, 63,14 min com erro de cerca de 1 s, parece muito confiável.

### 6.2 Verificação

**Contra o valor exato.** Este tanque tem primitiva:

In [ ]:
print(f"exato:  {T_exato / 60:.3f} min")
print(f"agente: {T_1000 / 60:.3f} min;  erro real = {T_exato - T_1000:.0f} s,  erro estimado pelo agente = {erro_est:.1f} s")

O erro real é de cerca de 40 s, **trinta vezes** maior que o estimado. Sem a primitiva, que um tanque real não tem, como perceber?

**Pela ordem observada.** A estimativa de Richardson supõe que o erro cai como $h^4$. Isso pode ser **medido**, refinando a malha:

In [ ]:
ns = np.array([100, 1000, 10_000, 100_000])
valores = np.array([tempo_esvaziamento(n) for n in ns])
for n, v in zip(ns, valores):
    print(f"n = {n:>6d}: T = {v / 60:.4f} min")
dif = np.abs(np.diff(valores))
print("diferenças sucessivas (s):", np.round(dif, 2))
print(f"ordem observada pelas diferenças: {np.log10(dif[0] / dif[1]):.2f} e {np.log10(dif[1] / dif[2]):.2f}")

A cada fator 10 em $n$, o resultado muda por um fator de só $10^{0{,}5} \approx 3$: a ordem é $1/2$, não 4. Com essa ordem, o fator $1/15$ da estimativa do agente está errado, e com ele a confiança no resultado.

### 6.3 Por quê

A fórmula de erro de Simpson {eq}`eq-simpson` contém $f^{(4)}$, que é infinita em $h = 0$ para um integrando com $1/\sqrt{h}$. E a "correção" `y[0] = 0` não é inofensiva: a área sob o pico do primeiro subintervalo, $\int_0^{\Delta h} \frac{A_0}{k\sqrt{h}}\,dh = \frac{2A_0\sqrt{\Delta h}}{k}$, é proporcional a $\sqrt{\Delta h}$, e é essencialmente ela que falta. Daí a ordem $1/2$. Refinar não resolve: com $n = 100\,000$ o erro ainda passa de 4 s, e para errar menos de 1 s seriam precisos cerca de 2 milhões de subintervalos.

### 6.4 A correção: mude a variável, não o passo

Com $h = u^2$, temos $dh = 2u\,du$ e $\sqrt{h} = u$:

$$
T = \int_0^{\sqrt{H}} \frac{2A(u^2)}{C_d\,a\sqrt{2g}}\,du .
$$

O $\sqrt{h}$ do denominador desaparece. O novo integrando é suave; aqui, é até um polinômio de grau 4 em $u$, que Gauss com 3 pontos integra **exatamente**:

In [ ]:
integrando_u = lambda u: 2 * area(u**2) / k
for n in [2, 3, 5]:
    T_g = gauss_legendre(integrando_u, 0.0, np.sqrt(H), n)
    print(f"Gauss com {n} pontos, depois de h = u²: {T_g / 60:.6f} min  (erro {T_g - T_exato:+.1e} s)")
print(f"Simpson n = 1000, depois de h = u²:      {simpson(integrando_u, 0.0, np.sqrt(H), 1000) / 60:.6f} min")
T_quad, est_quad = integrate.quad(integrando, 0.0, H)
print(f"scipy.integrate.quad na forma original:  {T_quad / 60:.6f} min  (erro estimado {est_quad:.0e} s)")

Três avaliações do integrando bastam. O `quad` também acerta, porque o QUADPACK detecta e extrapola singularidades nas pontas, mas a mudança de variável funciona com qualquer regra e com uma tabela de arqueação medida.

> **Lição.** Uma estimativa de erro por Richardson é tão boa quanto a ordem que ela supõe. **Meça a ordem** refinando a malha antes de confiar numa barra de erro. Quando o integrando tem uma singularidade, **mude a variável** para removê-la: é mais barato e mais preciso que qualquer refinamento. E desconfie de "correções" como `y[0] = 0`, que escondem um problema em vez de resolvê-lo.

## 7. Quando falha

### 7.1 Um pico estreito num intervalo longo

Um sensor de força registra o impacto de uma peça durante 1 s. O pulso dura alguns milissegundos e acontece em $t = 700$ ms. O **impulso** é a integral da força:

In [ ]:
forca = lambda t_ms: 2000 * np.exp(-(((t_ms - 700) / 2) ** 2))           # N, t em ms
impulso_exato = 2000 * 2 * np.sqrt(np.pi) / 1000                          # N·s
val, est = integrate.quad(forca, 0, 1000)
print(f"quad em [0, 1000] ms: {val / 1000:.3e} N·s (erro estimado {est / 1000:.0e})   exato: {impulso_exato:.3f} N·s")
val2, _ = integrate.quad(forca, 0, 1000, points=[700])
print(f"quad avisado do pico (points=[700]): {val2 / 1000:.3f} N·s")

O `quad` devolve praticamente zero, com uma estimativa de erro minúscula e **sem nenhum aviso**. Os primeiros 21 pontos de Gauss-Kronrod caíram todos longe do pulso, onde a força é zero, e a rotina concluiu que a função é zero. Uma rotina adaptativa só refina onde **vê** dificuldade. Se você sabe onde estão os picos, diga a ela (`points=`), ou divida o intervalo. Desenhe sempre o integrando antes de integrar.

### 7.2 Descontinuidades

Uma carga distribuída que muda bruscamente num ponto (uma parede sobre uma laje, por exemplo) tem um salto. Com o salto fora dos nós, toda regra de Newton-Cotes perde a ordem:

In [ ]:
carga = lambda x: np.where(x < 1 / 3, 2.0, 5.0)        # kN/m, salto em x = 1/3
exato_carga = 2.0 / 3 + 5.0 * 2 / 3
ns = np.array([10, 20, 40, 80, 160])
e_t = [abs(trapezio(carga, 0, 1, n) - exato_carga) for n in ns]
e_s = [abs(simpson(carga, 0, 1, n) - exato_carga) for n in ns]
print(f"ordem observada: trapézio {ordem_observada(1 / ns, e_t):.2f}, Simpson {ordem_observada(1 / ns, e_s):.2f}")
partes = simpson(carga, 0, 1 / 3 - 1e-15, 2) + simpson(carga, 1 / 3, 1, 2)
print(f"dividindo no salto, Simpson com 2 + 2 subintervalos: erro = {abs(partes - exato_carga):.1e}")

As duas regras caem para ordem 1, com erros que oscilam conforme a posição do salto entre os nós. Dividir a integral no ponto de descontinuidade devolve a ordem e, aqui, a exatidão.

### 7.3 Dados com ruído

Quando os valores vêm de medidas, a regra de ordem alta deixa de ajudar. O erro do ruído na integral é da ordem de $\sigma\sqrt{(b - a)h}$, independentemente da regra. Repetindo 300 vezes um "ensaio" com ruído de 5 % sobre $\int_0^\pi \sin x\,dx = 2$:

In [ ]:
rng = np.random.default_rng(5)
sig = 0.05                                                   # ruído de 5 % do pico
for n in [10, 100, 1000]:
    x = np.linspace(0, np.pi, n + 1)
    e_t, e_s = [], []
    for _ in range(300):                                     # 300 repetições do "ensaio"
        y = np.sin(x) + rng.normal(0, sig, x.size)
        e_t.append(trapezio_dados(x, y) - 2)
        e_s.append(integrate.simpson(y, x=x) - 2)
    print(f"n = {n:4d}: erro típico (RMS) trapézio = {np.sqrt(np.mean(np.square(e_t))):.4f}, "
          f"Simpson = {np.sqrt(np.mean(np.square(e_s))):.4f};  previsão σ√((b−a)h) = {sig * np.sqrt(np.pi * np.pi / n):.4f}")

Com ruído, Simpson e o trapézio erram por quantidades parecidas, e mais pontos ajudam só como $\sqrt{h}$, uma média estatística. Integrar dados ruidosos é bem condicionado (o ruído se cancela em parte), ao contrário de derivar (capítulo 10), mas a ordem das regras não importa mais.

## Resumo

| Método | Exato para | Erro | Quando usar |
| --- | --- | --- | --- |
| Trapézio composto | retas | $O(h^2)$; exponencial para periódicas | dados tabelados; funções periódicas num período |
| Simpson 1/3 composto | cúbicas | $O(h^4)$ | dados igualmente espaçados e lisos |
| Romberg | | $O(h^{2k})$, cada nível + 2 ordens | funções suaves que se pode avaliar |
| Gauss-Legendre ($n$ pontos) | grau $2n - 1$ | exponencial para funções suaves | funções suaves; integrandos caros |
| Adaptativa (`quad`) | | controlado pela tolerância | padrão; diga onde estão picos e quinas |
| Mudança de variável | | remove singularidades | $1/\sqrt{x}$, intervalos infinitos |
| Monte Carlo | | $O(N^{-1/2})$ em qualquer dimensão | muitas dimensões; relate o erro padrão |

## Exercícios

**Conceituais (sem computador)**

1. Deduza a regra 1/3 de Simpson integrando a parábola de Lagrange pelos três nós, e mostre que ela é exata para $x^3$.
2. Encontre os nós e pesos da quadratura de Gauss com 2 pontos impondo que ela seja exata para $1, t, t^2, t^3$.
3. Mostre que a primeira coluna de Romberg, $(4T_{2n} - T_n)/3$, é a regra de Simpson com $2n$ subintervalos.
4. Para $\int_0^\infty e^{-x}/(1 + x^2)\,dx$, proponha uma mudança de variável que leve a um intervalo finito com integrando limitado.

**Verificação (com agente)**

5. Peça a um agente o tempo de esvaziamento de um tanque a partir de uma tabela de arqueação com 12 níveis irregulares (invente uma). Verifique se o código trata a singularidade em $h = 0$ e o espaçamento irregular (sugestão: interpole $A$ em função de $u = \sqrt{h}$).
6. Peça a um agente a estimativa por Monte Carlo do volume de uma esfera de raio 1 em 10 dimensões, com barra de erro. Verifique com a fórmula exata e repita com 20 sementes diferentes: a fração de vezes em que o valor exato cai dentro de $\pm 2$ erros padrão é próxima de 95 %?
7. Peça a um agente o valor eficaz (RMS) de uma corrente distorcida, $i(t) = 10\sin\omega t + 3\sin 3\omega t + \sin 5\omega t$, a partir de 64 amostras num período. Compare trapézio e Simpson com o valor exato $\sqrt{(100 + 9 + 1)/2}$ e explique o resultado com a seção 5.2.

**Projeto aberto**

8. **Esvaziamento com perda de carga.** Num tubo de descarga longo, a vazão depende do fator de atrito (capítulo 3), que depende da própria vazão. Monte a integral do tempo de esvaziamento com o fator de atrito calculado por Colebrook a cada nível e compare com Torricelli.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 21 e 22.
- P. J. Davis e P. Rabinowitz, *Methods of Numerical Integration*, 2ª ed., Academic Press, 1984.
- L. N. Trefethen e J. A. C. Weideman, "The exponentially convergent trapezoidal rule", *SIAM Review*, 56(3), 2014.
- R. Piessens e outros, *QUADPACK: A Subroutine Package for Automatic Integration*, Springer, 1983.